# Data prep

In [ ]:
import pandas as pd
import torch
from torch_geometric.data import Data
from rdkit import Chem
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import os

CSV_PATH = '../data/raw/smiles_selfies_full.csv'
SMILES_COLUMN = 'smiles' 
SUBSET_FRAC = 0.025 # Subset will have 2.5% of the data
RANDOM_STATE = 42

def smiles_to_graph(smiles):
    """
    Converting Smiles to graph object: Pytorch geometric,
    where nodes will be the atomic number
    and the edges will be the bond type
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None 

    node_features = []
    for atom in mol.GetAtoms():
        node_features.append([atom.GetAtomicNum()])
    x = torch.tensor(node_features, dtype=torch.float)

    edges_list = []
    edge_features = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        
        edges_list.extend([[i, j], [j, i]])
        
        bond_type = bond.GetBondTypeAsDouble()
        edge_features.extend([[bond_type], [bond_type]])

    if not edges_list: # single atom 
        edge_index = torch.empty((2, 0), dtype=torch.long)
        edge_attr = torch.empty((0, 1), dtype=torch.float)
    else:
        edge_index = torch.tensor(edges_list, dtype=torch.long).t().contiguous()
        edge_attr = torch.tensor(edge_features, dtype=torch.float)

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, smiles=smiles)

def process_and_split(df, output_dir, prefix=""):
    graphs = []
    for smiles in tqdm(df[SMILES_COLUMN]):
        graph = smiles_to_graph(smiles)
        if graph is not None:
            graphs.append(graph)
            
    print(f"[{prefix}] Successfuly created {len(graphs)} graphs from {len(df)} compunds")

    train_val, test_data = train_test_split(graphs, test_size=0.1, random_state=RANDOM_STATE)
    train_data, val_data = train_test_split(train_val, test_size=0.1111, random_state=RANDOM_STATE) 

    os.makedirs(output_dir, exist_ok=True)
    
    torch.save(train_data, os.path.join(output_dir, 'train.pt'))
    torch.save(val_data, os.path.join(output_dir, 'val.pt'))
    torch.save(test_data, os.path.join(output_dir, 'test.pt'))

df = pd.read_csv(CSV_PATH)
process_and_split(df, output_dir='../data/processed', prefix="FULL DATASET")
df_subset = df.sample(frac=SUBSET_FRAC, random_state=RANDOM_STATE)
process_and_split(df_subset, output_dir='../data/subset', prefix="SUBSET DATASET")



ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html